# 1 · Data

Stage 1 of the chain. Everything downstream is derived from what this notebook writes; nothing in it
depends on a model, a panel or a representation.

**Position** **`1_data`** → `drug_catalog` → `2_drug_selection` → `3_representations` → `4a` → `4b` → `5_evaluation`

| | |
|---|---|
| **reads** | Zenodo record `21807175` — CTRPv2 reprocessed by DrEval, MD5-verified |
| | `scRNAseq_SCP542/expression/CPM_data.txt` — 5.4 GB, genes × cells, counts per million |
| | `scRNAseq_SCP542/other/UMIcount_data.txt` — 3.5 GB, un-normalised, read for depth and mito only |
| | `scRNAseq_SCP542/metadata/Metadata.txt` — cell-level annotation, Kinker's program scores |
| **writes** | `<variant>/SCP542_CCLE.h5ad` — 53,513 cells × HVG genes, `.X` = CPM |
| | `processed/.../umi_qc_covariates.csv` — per-cell depth and mitochondrial fraction, cached |
| | `metadata/drevalpy_CTRPv2_zenodo_<record>/` — the pinned response table plus `provenance.json` |

Two variants are built, `hvg5000` and `all_genes`, differing only in the gene filter.

**Why not scGPT, targets or splits here.** Those are stage 3. The split is not cosmetic:
`2_drug_selection` needs exactly the cell-line roster this notebook writes and the response table it
fetches, so the drug panel can be built before any representation exists and without consulting a
model.

**Why the steps are functions.** Each is one call into `scripts/preprocessing/pipeline.py`, where
every step owns its guard and preconditions, so running them out of order fails loudly instead of
producing something subtly wrong.

In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.layout import DEFAULT_CTRP_SCORE, DEFAULT_VARIANT, PipelinePaths
from scripts.preprocessing import pipeline

# The one variant the pipeline runs on. The hvg1000/2000/3000 sweep is built in
# analysis/qc/, next to the only notebook that consumes it -- a stage everyone runs
# should not carry an off-by-default branch for one analysis (Selin, 12.08.2026).
VARIANT = DEFAULT_VARIANT

# Carried so the paths object is complete, but NOT used by this stage: fetch and convert
# are both target-agnostic. The score first bites in stage 3, which writes one targets
# h5ad per measure.
SCORE = DEFAULT_CTRP_SCORE


# ⚠️ SET TO True FOR R2 ON 13.08.2026, AND IT MUST GO BACK TO False AFTERWARDS.
#
# `convert` and `scgpt` refuse to replace an existing artifact, because everything downstream
# derives from them: a silent rebuild invalidates every representation and target while leaving them
# on disk looking current. That guard is the reason this constant exists, and True disarms it for
# every subsequent run of this notebook, not only the intended one.
#
# It is True because R2 IS the genuine rebuild -- every artifact under data/processed/ predates the
# code that now produces it. Returning it to False is part of closing R2; it is written here rather
# than only in TODO.md because this line is what a later reader will actually run.
OVERWRITE = True

# THE VARIANTS R1 DECIDED (Selin, 12.08.2026): hvg5000 AND all_genes -- not all five, not
# hvg5000 alone. hvg5000 is the default training variant and all_genes is what the report's
# full-transcriptome numbers rest on; hvg1000/2000/3000 keep their current artifacts and are
# re-embedded later as a top-up if the gene-set sweep is to be like-for-like (docs/TODO.md, R1).
#
# Parameterised 13.08.2026. Until then both drivers ran the single `DEFAULT_VARIANT`, so covering
# R1's decision meant running the notebook twice and remembering to edit a constant in between --
# which is how a variant silently goes missing from a rerun. The loop is the record of the decision.
VARIANTS_TO_RUN = ('hvg5000', 'all_genes')
PATHS = {v: PipelinePaths.build(None, v, SCORE) for v in VARIANTS_TO_RUN}

# `fetch` is variant-agnostic -- it writes into data/metadata/, shared by every variant -- so it
# takes one paths object rather than the loop.
paths = PATHS[VARIANT]
print(f'data root : {paths.data_root}')
print(f'convert   : {len(VARIANTS_TO_RUN)} variants -> {list(VARIANTS_TO_RUN)}')
print(f'variant   : {paths.variant} -> {paths.processed_dir}')
print(f'score     : {paths.score}  (unused here; stage 3 writes {paths.targets_h5ad.name})')

data root : /Users/selin/Desktop/OncoTox/data
convert   : 2 variants -> ['hvg5000', 'all_genes']
variant   : hvg5000 -> /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000
score     : auc_cc  (unused here; stage 3 writes SCP542_CCLE_scGPT_human_embeddings_with_targets_auc_cc.h5ad)


## A · The response table

| | |
|---|---|
| **in** | Zenodo record `layout.ZENODO_RESPONSE_RECORD`, two archives |
| **out** | `metadata/drevalpy_CTRPv2_zenodo_<record>/` — `CTRPv2.csv`, Cellosaurus, `provenance.json` |

CTRPv2's dose-response data as reprocessed by DrEval: raw measurements normalised against each
replicate's own no-drug control, then every curve re-fitted with CurveCurator. That folds replicate
disagreement into the fit rather than averaging replicates before fitting, which their Methods argue
*"leads to inaccurate or misleading drug response measures in the case of large discrepancies between
replicates"*.

**Why the record is pinned.** `drevalpy`'s own loader resolves the concept DOI to whatever is latest,
so the data it returns changes underneath you. Pinning one record fixes the target; the cache
directory carries its number, so an older copy stays visible on disk rather than being silently
overwritten. Bumping it is a **target change** — every number downstream has to be re-derived.

Both archives are MD5-verified against the record's published checksums and refuse to proceed on a
mismatch, so the step is safe to re-run: a cached archive matching its MD5 is neither re-downloaded
nor re-extracted. This is the only step that touches the network.

In [2]:
response_csv = pipeline.fetch(paths)
response_csv

[fetch] Zenodo record 21807175


Zenodo record 21807175 -- 'Dataset for drevalpy', DOI 10.5281/zenodo.21807175, published 2026-08-05


  CTRPv2.zip: cached, MD5 verified -- skipping download
  meta.zip: cached, MD5 verified -- skipping download
Cached at /Users/selin/Desktop/OncoTox/data/metadata/drevalpy_CTRPv2_zenodo_21807175


PosixPath('/Users/selin/Desktop/OncoTox/data/metadata/drevalpy_CTRPv2_zenodo_21807175/CTRPv2/CTRPv2.csv')

In [3]:
import json

# What was actually retrieved, read from the data rather than from the code that fetched it --
# fetch() writes this alongside the archives so the version is legible from the artifact.
prov = json.loads((paths.drevalpy_dir / 'provenance.json').read_text())
for k in ('zenodo_record', 'doi', 'publication_date', 'retrieved'):
    print(f'{k:18s} {prov[k]}')
print(f'{"files":18s} {", ".join(prov["files"])}')

zenodo_record      21807175
doi                10.5281/zenodo.21807175
publication_date   2026-08-05
retrieved          2026-08-13
files              CTRPv2.zip, meta.zip


## B · The expression matrix

| | |
|---|---|
| **in** | `CPM_data.txt` (genes × cells), `Metadata.txt`, `UMIcount_data.txt` |
| **out** | `<variant>/SCP542_CCLE.h5ad` — cells × genes, `.X` = CPM, `obs` = metadata + QC covariates |

SCP542 is distributed as counts per million, so library-size normalisation has already been applied
once, to the full gene matrix, which is where it belongs. This step does not normalise again.

**Why the transform is applied to a copy.** Gene selection ranks on the authors' own quantification,
$E_{ij} = \log_2(1 + \mathrm{CPM}_{ij}/10)$ — the divisor is theirs, argued from the average UMI
count per cell being below $10^5$, without which the difference between a detected and an undetected
gene is inflated. The saved `.X` keeps the original CPM values, because scGPT reads only the *order*
of a cell's values and any strictly increasing map leaves that order unchanged.

**Why depth and mitochondrial fraction come from a different file.** They cannot be recovered from
`.X`: CPM has divided library size out, and only 4 of 13 `MT-` genes survive HVG selection. Both are
read from the un-normalised UMI matrix over the **full** gene set and joined by barcode — a total
over the variable genes is not depth but depth times the share of a cell's counts falling in that
set, and that share is biological. They are the covariates `4b`'s confound veto is defined on.

**Why `convert` refuses to overwrite.** Every representation and target downstream derives from this
file, so replacing it silently invalidates them while leaving them on disk looking current.

In [4]:
# One call per variant. `convert` guards each output itself, so a variant already built raises
# rather than being rebuilt silently -- see OVERWRITE above.
raw_h5ad = {v: pipeline.convert(PATHS[v], overwrite=OVERWRITE) for v in VARIANTS_TO_RUN}
raw_h5ad

[convert] hvg5000: top-5000 HVGs
Loading expression matrix... (this may take a few minutes and require high RAM)


Loading metadata...
Aligning metadata with expression data...
[qc] reading cached UMI covariates from umi_qc_covariates.csv
[qc] join check: genes detected vs obs['Genes_expressed'] r=1.0000
[qc] added ['total_counts', 'pct_counts_mt'] | median depth 16,733 | median mito 5.61%
Annotating current HGNC symbols...


  hgnc_symbol: 1,129 of 22,722 rows renamed to their current symbol, 23 renames withheld as collisions
Selecting top 5000 highly variable genes...


  Gene count: 22722 -> 5000
Saving to /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000/SCP542_CCLE.h5ad...


/Users/selin/PycharmProjects/OncoTox/.venv/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


...

Loading metadata...
Aligning metadata with expression data...
[qc] reading cached UMI covariates from umi_qc_covariates.csv
[qc] join check: genes detected vs obs['Genes_expressed'] r=1.0000
[qc] added ['total_counts', 'pct_counts_mt'] | median depth 16,733 | median mito 5.61%
Annotating current HGNC symbols...


  hgnc_symbol: 1,129 of 22,722 rows renamed to their current symbol, 23 renames withheld as collisions
Saving to /Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/all_genes/SCP542_CCLE.h5ad...


/Users/selin/PycharmProjects/OncoTox/.venv/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
  return func(*args, **kwargs)


Success! Created AnnData object: AnnData object with n_obs × n_vars = 53513 × 22722
    obs: 'Cell_line', 'Pool_ID', 'Cancer_type', 'Genes_expressed', 'Discrete_cluster_minpts5_eps1.8', 'Discrete_cluster_minpts5_eps1.5', 'Discrete_cluster_minpts5_eps1.2', 'CNA_subclone', 'SkinPig_score', 'EMTI_score', 'EMTII_score', 'EMTIII_score', 'IFNResp_score', 'p53Sen_score', 'EpiSen_score', 'StressResp_score', 'ProtMatu_score', 'ProtDegra_score', 'G1/S_score', 'G2/M_score', 'total_counts', 'pct_counts_mt'
    var: 'hgnc_symbol'


{'hvg5000': PosixPath('/Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/hvg5000/SCP542_CCLE.h5ad'),
 'all_genes': PosixPath('/Users/selin/Desktop/OncoTox/data/processed/scRNAseq_SCP542/all_genes/SCP542_CCLE.h5ad')}

In [5]:
import anndata as ad

# The distributed gene count used to be written here as a literal ("of SCP542's 22,722
# distributed"). Removed 13.08.2026: convert() already prints `Gene count: <before> -> <after>`
# from the matrix it just read, one cell above, so the literal was a second copy of a number the
# run computes -- and the copy is the one that goes stale when the source or the HVG step moves.
#
# obs columns are printed too (13.08.2026): total_counts and pct_counts_mt are written here by
# convert, from the raw UMI matrix, and their absence is what makes 4b's confound veto unevaluable
# -- so this is the cell where a missing one should be noticed, not four stages later.
for v in VARIANTS_TO_RUN:
    a = ad.read_h5ad(raw_h5ad[v], backed='r')      # backed: .X stays on disk
    lines = a.obs['Cell_line'].astype(str).str.split('_').str[0].nunique()
    qc = [c for c in ('total_counts', 'pct_counts_mt') if c in a.obs.columns]
    print(f'--- {v} ---')
    print(f'cells      : {a.n_obs:,}')
    print(f'genes kept : {a.n_vars:,}')
    print(f'cell lines : {lines}')
    print(f'.X         : {a.X.dtype}, still CPM -- max {a.X[:200].max():.1f}')
    print(f'var cols   : {list(a.var.columns)}')
    print(f'UMI qc     : {qc if qc else "MISSING -- 4b stage 6 cannot run"}')
    a.file.close()

--- hvg5000 ---
cells      : 53,513
genes kept : 5,000
cell lines : 198
.X         : float64, still CPM -- max 90079.0
var cols   : ['hgnc_symbol']
UMI qc     : ['total_counts', 'pct_counts_mt']
--- all_genes ---
cells      : 53,513
genes kept : 22,722
cell lines : 198
.X         : float64, still CPM -- max 90079.0
var cols   : ['hgnc_symbol']
UMI qc     : ['total_counts', 'pct_counts_mt']
